# 01 — HMM Regime EDA: BTC vs Gold

This notebook performs exploratory data analysis on BTC and XAUUSD, then fits the HMM to visualise the latent regime sequence overlaid on price.

**Steps**:
1. Load cached OHLCV (or fetch live)
2. Feature engineering
3. BIC-based regime count selection
4. HMM fit + Viterbi decode
5. Price series coloured by regime
6. Regime statistics table

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from repo root

import asyncio
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from config import settings
settings.ensure_dirs()

print('Environment ready.')

In [ ]:
# ── Fetch data ──────────────────────────────────────────────────────────────
from data.feed import fetch_all

data = asyncio.run(fetch_all(timeframe='1d', since='2020-01-01', until='2024-12-31'))
btc_raw  = data['BTC']
gold_raw = data['XAUUSD']

print(f"BTC rows:  {len(btc_raw):,}   |  first: {btc_raw.index[0].date()}  last: {btc_raw.index[-1].date()}")
print(f"Gold rows: {len(gold_raw):,}  |  first: {gold_raw.index[0].date()}  last: {gold_raw.index[-1].date()}")

In [ ]:
# ── Feature engineering ─────────────────────────────────────────────────────
from data.features import FeatureEngineer

fe = FeatureEngineer()
btc_feat  = fe.transform(btc_raw)
gold_feat = fe.transform(gold_raw)

btc_obs  = fe.get_hmm_observations(btc_feat)
gold_obs = fe.get_hmm_observations(gold_feat)

print('Feature shapes:', btc_obs.shape, gold_obs.shape)

In [ ]:
# ── BIC model selection ─────────────────────────────────────────────────────
from models.hmm_regimes import select_optimal_n_regimes
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, obs, label in zip(axes, [btc_obs, gold_obs], ['BTC', 'XAUUSD']):
    best_k, bic_scores = select_optimal_n_regimes(obs, k_range=(2, 7))
    ks = list(range(2, 2 + len(bic_scores)))
    ax.plot(ks, bic_scores, marker='o', color='steelblue')
    ax.axvline(best_k, color='crimson', linestyle='--', label=f'Optimal K={best_k}')
    ax.set_title(f'{label} — BIC by K')
    ax.set_xlabel('Number of Regimes (K)')
    ax.set_ylabel('BIC')
    ax.legend()

plt.tight_layout()
plt.show()

print(f'BTC optimal K={best_k}')

In [ ]:
# ── Fit HMMs ────────────────────────────────────────────────────────────────
from models.hmm_regimes import HMMRegimeDetector, select_optimal_n_regimes

btc_k, _  = select_optimal_n_regimes(btc_obs, k_range=(2, 6))
gold_k, _ = select_optimal_n_regimes(gold_obs, k_range=(2, 6))

btc_hmm  = HMMRegimeDetector(n_regimes=btc_k).fit(btc_obs)
gold_hmm = HMMRegimeDetector(n_regimes=gold_k).fit(gold_obs)

btc_regimes  = btc_hmm.predict_regimes(btc_obs)
gold_regimes = gold_hmm.predict_regimes(gold_obs)

print('BTC regime labels: ', btc_hmm._regime_labels)
print('Gold regime labels:', gold_hmm._regime_labels)

In [ ]:
# ── Plot: Price coloured by HMM regime ──────────────────────────────────────
PALETTE = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6', '#1abc9c']

def plot_regimes(ax, price_series, regimes, labels, title):
    unique_regimes = sorted(set(regimes))
    for r in unique_regimes:
        mask = regimes == r
        color = PALETTE[r % len(PALETTE)]
        ax.fill_between(price_series.index, price_series.min(), price_series.max(),
                        where=mask, alpha=0.25, color=color)
    ax.plot(price_series.index, price_series.values, color='black', linewidth=0.7, zorder=5)
    
    patches = [mpatches.Patch(color=PALETTE[r % len(PALETTE)], label=f'{r}: {labels.get(r,"")}')
               for r in unique_regimes]
    ax.legend(handles=patches, loc='upper left', fontsize=8)
    ax.set_title(title)
    ax.set_ylabel('Price')

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)

plot_regimes(axes[0], btc_feat['close'], btc_regimes, btc_hmm._regime_labels, 'BTC/USDT — HMM Regimes')
plot_regimes(axes[1], gold_feat['close'], gold_regimes, gold_hmm._regime_labels, 'XAUUSD (Gold) — HMM Regimes')

plt.tight_layout()
plt.savefig('../data/cache/hmm_regimes_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# ── Regime statistics table ──────────────────────────────────────────────────
def regime_stats(feat_df, regimes, labels):
    feat_df = feat_df.copy()
    feat_df['regime'] = regimes
    feat_df['label'] = feat_df['regime'].map(labels)
    stats = feat_df.groupby('label').agg(
        n_bars=('log_ret', 'count'),
        mean_log_ret=('log_ret', 'mean'),
        std_log_ret=('log_ret', 'std'),
        mean_realised_vol=('realised_vol', 'mean'),
        mean_atr_norm=('atr_norm', 'mean'),
    ).round(6)
    stats['hit_rate_pct'] = (stats['n_bars'] / stats['n_bars'].sum() * 100).round(1)
    return stats

print('=== BTC Regime Statistics ===')
display(regime_stats(btc_feat, btc_regimes, btc_hmm._regime_labels))

print('\n=== Gold Regime Statistics ===')
display(regime_stats(gold_feat, gold_regimes, gold_hmm._regime_labels))

In [ ]:
# ── Rolling correlation BTC vs Gold ─────────────────────────────────────────
common = btc_feat.index.intersection(gold_feat.index)
btc_r  = btc_feat.loc[common, 'log_ret']
gold_r = gold_feat.loc[common, 'log_ret']

rolling_corr = btc_r.rolling(30).corr(gold_r)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rolling_corr.index, rolling_corr.values, color='darkorange', linewidth=0.9)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.axhline(0.5, color='crimson', linewidth=0.5, linestyle=':', label='Hedge threshold (+0.5)')
ax.axhline(-0.5, color='royalblue', linewidth=0.5, linestyle=':', label='Hedge threshold (-0.5)')
ax.set_title('30-day Rolling Correlation: BTC vs XAUUSD (log-returns)')
ax.set_ylabel('Pearson r')
ax.legend()
plt.tight_layout()
plt.show()